# 1. Data Exploration & Preprocessing Pipeline
**Project**: House Price Prediction  
**Goal**: Clean raw data, filter synthetic rows/outliers, engineer features, encode categoricals, and prepare train/test splits without data leakage.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Raw Dataset & Filter Synthetic Rows
We load `House_Prices.csv` and filter out synthetic unlabeled test rows (rows 1460 to 2918) and extreme area outliers (`GrLivArea >= 4000`).

In [2]:
data_path = os.path.join("..", "data", "raw", "House_Prices.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "raw", "House_Prices.csv")

df_raw = pd.read_csv(data_path)
print("Raw dataset shape:", df_raw.shape)

# Filter out synthetic target rows (rows 1460 to 2918)
df_clean = df_raw.iloc[:1460].copy()

# Remove extreme high-leverage outliers (GrLivArea > 4000 sq ft)
df_clean = df_clean[df_clean["GrLivArea"] < 4000].reset_index(drop=True)
print("Clean ground-truth dataset shape:", df_clean.shape)

Raw dataset shape: (2919, 81)
Clean ground-truth dataset shape: (1456, 81)


## 3. Handle Missing Values

In [3]:
# 1. Structural Categorical Columns -> 'None'
cat_none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "MasVnrType"
]
for col in cat_none_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("None")

# 2. Structural Numerical Columns -> 0
num_zero_cols = [
    "GarageArea", "GarageCars", "BsmtFinSF1", "BsmtFinSF2",
    "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath",
    "MasVnrArea", "GarageYrBlt"
]
for col in num_zero_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

# 3. LotFrontage -> Neighborhood Median
df_clean["LotFrontage"] = df_clean.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))
df_clean["LotFrontage"] = df_clean["LotFrontage"].fillna(df_clean["LotFrontage"].median())

# 4. General Categoricals -> Mode
gen_cat_cols = ["MSZoning", "Utilities", "Exterior1st", "Exterior2nd", "Electrical", "Functional", "SaleType"]
for col in gen_cat_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Remaining missing values:", df_clean.isnull().sum().sum())

Remaining missing values: 0


## 4. Ordinal Quality Encoding & Domain Feature Engineering

In [4]:
# Ordinal Mapping
qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ord_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
for col in ord_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].map(qual_map).fillna(0)

finish_map = {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3}
if "GarageFinish" in df_clean.columns:
    df_clean["GarageFinish"] = df_clean["GarageFinish"].map(finish_map).fillna(0)

exposure_map = {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}
if "BsmtExposure" in df_clean.columns:
    df_clean["BsmtExposure"] = df_clean["BsmtExposure"].map(exposure_map).fillna(0)

# Domain Feature Engineering
df_clean['TotalArea'] = df_clean['TotalBsmtSF'] + df_clean['1stFlrSF'] + df_clean['2ndFlrSF']
df_clean['TotalBathrooms'] = df_clean['FullBath'] + (0.5 * df_clean['HalfBath']) + df_clean['BsmtFullBath'] + (0.5 * df_clean['BsmtHalfBath'])
df_clean['HouseAge'] = np.maximum(0, df_clean['YrSold'] - df_clean['YearBuilt'])
df_clean['YearsSinceRemodel'] = np.maximum(0, df_clean['YrSold'] - df_clean['YearRemodAdd'])
porch_cols = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
df_clean['TotalPorchArea'] = df_clean[porch_cols].sum(axis=1)
df_clean['HasGarage'] = (df_clean['GarageArea'] > 0).astype(int)
df_clean['HasFireplace'] = (df_clean['Fireplaces'] > 0).astype(int)
df_clean['HasBasement'] = (df_clean['TotalBsmtSF'] > 0).astype(int)

print("Feature engineering and ordinal mapping completed successfully!")

Feature engineering and ordinal mapping completed successfully!


## 5. One-Hot Encoding, Train/Test Split, and Feature Scaling

In [5]:
X = df_clean.drop(columns=["Id", "SalePrice"])
y = df_clean["SalePrice"]

# One-hot encoding nominal categorical features
nominal_cols = [col for col in X.columns if not pd.api.types.is_numeric_dtype(X[col])]
X_encoded = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Standardize continuous numerical columns only (leaving 0/1 binary columns unscaled)
numeric_cols = [col for col in X_train.columns if pd.api.types.is_numeric_dtype(X_train[col])]
continuous_cols = [c for c in numeric_cols if X_train[c].nunique() > 2]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape: ", X_test_scaled.shape)

X_train_scaled shape: (1164, 231)
X_test_scaled shape:  (292, 231)


## 6. Save Processed Datasets

In [6]:
output_dir = os.path.join("..", "data", "processed")
if not os.path.exists(output_dir):
    output_dir = os.path.join("data", "processed")
os.makedirs(output_dir, exist_ok=True)

X_train_scaled.to_csv(os.path.join(output_dir, "X_train.csv"), index=False)
X_test_scaled.to_csv(os.path.join(output_dir, "X_test.csv"), index=False)
pd.DataFrame({"SalePrice": y_train}).to_csv(os.path.join(output_dir, "y_train.csv"), index=False)
pd.DataFrame({"SalePrice": y_test}).to_csv(os.path.join(output_dir, "y_test.csv"), index=False)

train_full = X_train_scaled.copy()
train_full["SalePrice"] = y_train.values
train_full.to_csv(os.path.join(output_dir, "train_processed.csv"), index=False)

test_full = X_test_scaled.copy()
test_full["SalePrice"] = y_test.values
test_full.to_csv(os.path.join(output_dir, "test_processed.csv"), index=False)

print("All processed datasets saved successfully to data/processed/")

All processed datasets saved successfully to data/processed/
